In [ ]:
import ee
import geopandas as gpd
from geesat import geogee, geosen
import geesat
# geesat.authenticate_gee()
ee.Authenticate()
ee.Initialize(project='ee-tuyenrss')

In [4]:
# Example of area of interest (AOI) in Germany
bbox = [9.84375   , 47.5172007 , 10.1953125 , 47.75409798]
# Create an Earth Engine geometry object for the AOI
aoi = ee.Geometry.Rectangle(bbox)
# if you have geopandas dataframe, you can convert it to an Earth Engine geometry object like this:
# gdf = gpd.read_file("path_to_your_shapefile.shp")
# aoi = geesat.gdf_to_ee(gdf)

# 1. Landsat dataset

- **Generate cloud-free monthly LST in degree**

In [ ]:
lst = geogee.generate_landsat_lst(aoi, start_date='1990-01-01', end_date='1990-12-31')
# make monthly composite
monthly_lst = geogee.generate_monthly_composite(lst, aggregate_method='mean')
print(f"Number of images in the original collection: {lst.size().getInfo()}")
print(f"Number of images in the monthly composite collection: {monthly_lst.size().getInfo()}")

- **Generate cloud-free surface reflectance and vegetation-related indices**

In [ ]:
# Calculate predefined Landsat indices from Landsat 8 and 9 data
landsat_indices = geogee.generate_landsat_indices(aoi=aoi, start_date='2020-01-01', end_date='2020-12-31')
print(landsat_indices.first().bandNames().getInfo())

# 2. Generate cloud-free Sentinel-2 data

- **Generate cloud-free data**

In [ ]:
sen2data = geogee.generate_sen2_cloud_mask(aoi, start_date='2020-01-01', end_date='2020-12-31', cloud_filter=50)
print(f"Number of images in the original Sentinel-2 collection: {sen2data.size().getInfo()}")

- **Generate monthly cloud-free composite**

In [ ]:
# make monthly composite
monthly_sen2data = geogee.generate_monthly_composite(sen2data, aggregate_method='median')
print(f"Number of images in the monthly composite Sentinel-2 collection: {monthly_sen2data.size().getInfo()}")

- **Generate n-day cloud-free composite**

In [ ]:
# make n-day composite 
composite = geogee.generate_nday_composite(sen2data, aggregate_method='median', n_days=16)
print(f"Number of images in the 16-day composite Sentinel-2 collection: {composite.size().getInfo()}")

# 3. Load Sentinel-1 data and generate vh/vv and RVI (Radar vegetation Index) indices

In [ ]:
sen1data = ee.ImageCollection("COPERNICUS/S1_GRD").filterBounds(aoi).filterDate('2020-01-01', '2020-12-31')
sen1indices = geogee.generate_sentinel1_indices(sen1data)
print(f"Number of images in the Sentinel-1 indices collection: {sen1indices.size().getInfo()}")
print(f"Band names in the Sentinel-1 indices collection: {sen1indices.first().bandNames().getInfo()}")

# 4. Extract raster values by vector data

In [ ]:
# create 30 random points within the AOI
points = ee.FeatureCollection.randomPoints(aoi, 30)
print(f"Number of random points created: {points.size().getInfo()}")
# Create 30 polygons around the points with a buffer of 500 meters
polygons = points.map(lambda point: point.buffer(500).bounds())
print(f"Number of random polygons created: {polygons.size().getInfo()}")

- **Extract raster values by points**

In [ ]:
# Example extract LST values at the random points
lst = geogee.generate_landsat_lst(aoi, start_date='2025-06-01', end_date='2025-10-31')

lst_df = geogee.extract_raster_values_by_points(points, col=lst, scale=30, to_gdf=True).drop(columns=["id"])
lst_df.head() # NANs means cloud-masked pixels or missing data.

In [ ]:
# Another example extract Sentinel-2 data
sen2data = geogee.generate_sen2_cloud_mask(aoi, start_date='2025-06-01', end_date='2025-10-31', cloud_filter=50)
sen2df = geogee.extract_raster_values_by_points(points, col=sen2data, scale=30, to_gdf=True).drop(columns=["id"])
sen2df.head() # NANs means cloud-masked pixels or missing data.

- **Extract raster values by polygons**

In [ ]:
lst_df = geogee.extract_raster_values_by_polygons(
    col=lst,
    polygon=polygons,
    scale=30,
    aggregate_method='mean',
    to_gdf=True
)
lst_df.head() # NANs means cloud-masked pixels or missing data.

In [ ]:
# Another example extract Sentinel-2 data
sen2data = geogee.generate_sen2_cloud_mask(aoi, start_date='2025-06-01', end_date='2025-10-31', cloud_filter=50)
sen2df = geogee.extract_raster_values_by_polygons(
    col=sen2data,
    polygon=polygons,
    scale=30,
    aggregate_method='mean',
    to_gdf=True
)
sen2df.head() # NANs means cloud-masked pixels or missing data.

# 5. Generate water occurance using sentinel-1

In [5]:
water_occurrence = geosen.generate_water_occurrence(roi=aoi, start_date='2025-01-01', end_date='2025-12-31')
# print(water_occurrence.bandNames().getInfo())
print(water_occurrence.bandNames().getInfo())

['water_occurrence']


In [11]:
# def generate_hls_data(roi,
#                     start_date='2020-01-01',
#                     end_date='2020-12-31'     ,
#                    cloud_filter=40,
#                    mask_cirrus=True,
#                    mask_high_aerosol=True,
#                    buffer_m=0,
#                    sensors=('S2', 'L8')
#                    ):
#     """
#     Harmonized Landsat 8/9 + Sentinel-2 (HLS v2.0, 30 m) with Fmask cloud masking.

#     Parameters
#     ----------
#     roi : ee.Geometry or ee.FeatureCollection
#     start_date, end_date : 'YYYY-MM-DD' (end date is exclusive)
#     cloud_filter : max scene CLOUD_COVERAGE in percent; None to skip
#     mask_cirrus : also mask Fmask cirrus bit
#     mask_high_aerosol : also mask pixels with high aerosol level (haze)
#     buffer_m : grow cloud/shadow mask by this many meters (e.g. 30-60)
#     sensors : subset of ('S2', 'L8'), to use one or both products
#     add_indices : add NDVI, EVI2, NDMI, NBR bands

#     Returns
#     -------
#     ee.ImageCollection with bands BLUE, GREEN, RED, NIR, SWIR1, SWIR2
#     (reflectance 0-1) plus a 'sensor' property.
#     """
#     names = ['BLUE', 'GREEN', 'RED', 'NIR', 'SWIR1', 'SWIR2']

#     def _mask(img):
#         f = img.select('Fmask')
#         bad = (f.bitwiseAnd(1 << 1)                 # cloud
#                .Or(f.bitwiseAnd(1 << 2))            # adjacent to cloud/shadow
#                .Or(f.bitwiseAnd(1 << 3)))           # cloud shadow
#         if mask_cirrus:
#             bad = bad.Or(f.bitwiseAnd(1 << 0))      # cirrus
#         if mask_high_aerosol:
#             aerosol = f.rightShift(6).bitwiseAnd(3)  # bits 6-7: 3 = high
#             bad = bad.Or(aerosol.eq(3))
#         bad = bad.gt(0)
#         if buffer_m > 0:
#             bad = bad.focalMax(buffer_m, 'circle', 'meters')
#         return img.updateMask(bad.Not())


#     def _build(asset, bands_in, sensor):
#         col = (ee.ImageCollection(asset)
#                .filterBounds(roi)
#                .filterDate(start_date, end_date))
#         if cloud_filter is not None:
#             col = col.filter(ee.Filter.lt('CLOUD_COVERAGE', cloud_filter))

#         def _prep(img):
#             img = _mask(img).select(bands_in, names)       # mask first, then rename
#             return (img.copyProperties(img, ['system:time_start'])
#                        .set('sensor', sensor)
#                        .set('system:time_start', ee.Image(img).get('system:time_start')))
#         return col.map(_prep)

#     cols = []
#     if 'S2' in sensors:
#         # NIR narrow (B8A) matches the Landsat OLI NIR band
#         cols.append(_build('NASA/HLS/HLSS30/v002',
#                            ['B2', 'B3', 'B4', 'B8A', 'B11', 'B12'], 'S2'))
#     if 'L8' in sensors:
#         cols.append(_build('NASA/HLS/HLSL30/v002',
#                            ['B2', 'B3', 'B4', 'B5', 'B6', 'B7'], 'L8'))

#     out = cols[0]
#     for c in cols[1:]:
#         out = out.merge(c)
#     return out.sort('system:time_start')

In [2]:

poly = gpd.read_file(r"C:\Users\tuyen\Downloads\upstream_aoi.geojson")
aoi = geesat.gdf_to_ee(poly)

In [35]:
# test = generate_hls_data(roi=aoi, start_date='2016-10-01', end_date='2017-05-30', cloud_filter=60)
# print(test.first().bandNames().getInfo())
sen2data = geogee.generate_sen2sr_cloud_mask(
    aoi=aoi,
    start_date='2024-11-01',
    end_date='2025-03-30')
print(sen2data.size().getInfo())

44


In [36]:
import geemap 
Map = geemap.Map()
Map.addLayer(sen2data.median().clip(aoi), {'bands': ['B4', 'B3', 'B2'], 'min': 0, 'max': 2000}, 'First Image')
Map.centerObject(aoi, 12)
Map

Map(center=[-7.471989005015427, 145.175599504549], controls=(WidgetControl(options=['position', 'transparent_b…